## **K-means Clustering Code**

### **분석 과정 및 활용 기술**

본 분석은 제공된 **이미지 특징 벡터(`image_features.npy`)**와 **정답 라벨(`image_labels.npy`)**을 기반으로, 다음의 네 단계로 구성된 파이프라인을 통해 수행되었다.

- #### **1단계: 특징 데이터 로딩 & 데이터 정형화**  
    - **활용 기술:** *VGG16 (Convolutional Neural Network)*  
    - **주요 역할:**
        - 사전 학습된 VGG16 모델을 통해 이미지로부터 추출된 고차원 특징 벡터(Feature Vector)를 불러온다. 이 작업은 전이 학습(Transfer Learning)의 일종이다.
        - 이미 수백만 장의 사진들의 패턴과 특징들을 사전에 학습하고 이해한 VGG16 딥러닝 모델을 활용하여 각각의 사진들을 분석하고, 그에 맞는 고차원의 벡터로 전환시킨 과정.
        - 이를 통해 이미지 전처리 및 특징 추출에 소요되는 연산을 생략하고, 효율적으로 데이터 분석 단계로 진입할 수 있다.
        - 비유로 표현하자면, 이 단계는 컴퓨터가 이해할 수 없는 '이미지'를 컴퓨터가 직접 읽고 이해할 수 있는 한 권의 책으로 변환하는 과정이다.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# 딥러닝 및 머신러닝 라이브러리 임포트
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# load_and_preprocess_images 함수와 extract_features 함수는 삭제

# --- 메인 실행 코드 ---
if __name__ == "__main__":
    
    # 1. 데이터 준비 (새로운 방식)
    print("1. .npy 파일로부터 특징 벡터와 라벨을 로드합니다...")
    # 현재 스크립트 파일이 있는 폴더의 절대 경로를 가져오기
    script_path = os.path.dirname(os.path.abspath(__file__))

    # 파일의 전체 경로(절대 경로)를 만들오기
    features_path = os.path.join(script_path, 'image_features.npy')
    labels_path = os.path.join(script_path, 'image_labels.npy')

    # 전체 경로를 사용해서 파일을 불러오기.
    features = np.load(features_path)
    true_labels = np.load(labels_path)
    
    print(f"총 {len(features)}개의 특징 벡터를 로드했습니다.")
    print(f"로드된 특징 벡터의 형태(shape): {features.shape}")

    # 2. VGG16 모델 준비 및 특징 추출 단계는 건너뛰기    
   

- #### **2단계: 차원 축소**  
    - **활용 기술:** *PCA (Principal Component Analysis, 주성분 분석)*  
    - **주요 역할:**
        - 고차원 특징 벡터에 포함된 잡음(noise)을 제거하고, 데이터의 핵심 정보(주성분)만을 압축한다.
        - 이미지 데이터들 가운데서 서로를 가장 잘 구분지을 수 있는 주요 성분 (예를 들어 색깔이나, 옷의 핏 등등)을 찾아내고, 그것들만 남긴 가볍고 깨끗한 데이터로 전환하는 작업
        - 이를 통해 **K-means 클러스터링의 계산 효율성 및 안정성**을 동시에 향상시킨다.
        - 데이터 정형화가 “컴퓨터가 읽을 수 있는 책”으로 바꾸는 작업이라면, 차원 축소는, 그 책을 핵심 내용만 담은 요약본으로 만드는 작업이라고 볼 수 있다.

In [ ]:
    # 3. PCA로 차원 축소 
    print("\n3. PCA를 사용하여 특징 벡터의 차원을 축소합니다...")
    # K-means의 성능과 안정성을 위해 주성분 50개로 차원을 축소
    pca = PCA(n_components=50, random_state=42)
    # 바로 features 변수를 사용
    features_pca = pca.fit_transform(features)
    print(f"PCA로 축소된 특징 벡터의 형태: {features_pca.shape}")

- #### **3단계: 군집화**
    - **기술:** K-means 클러스터링
    - **주요 역할:**
        - PCA로 압축된 데이터를 기반으로, **어떠한 정답 정보도 주지 않은 상태에서(비지도 학습)** 순수하게 데이터의 시각적 유사성만을 이용해 4개의 그룹(클러스터)으로 자동 분류했음.
    - **분류 방식**
        1. 수 많은 데이터들 가운데, 4개의 중심점을 무작위로 설정
        2. 각각의 데이터가 4개의 중심점 중에 가장 가까운 곳에 한 팀이 되어서 분리됨
        3. 각각의 데이터가 4개의 그룹으로 만들어지면, 그 팀들 가운데 가장 중앙에 위치한 데이터를 다시 중심점으로 설정
        4. 그 중심점을 기준으로 똑같이 분류
        5. 이러한 과정을 여러 번 반복

In [ ]:
    # 4. K-means 클러스터링 실행
    print("\n4. K-means 클러스터링을 진행합니다...")
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(features_pca)

- #### **4단계: 결과 시각화**
    - **기술:** t-SNE
    - **주요 역할:**
        - 고차원 공간에 존재하는 데이터 포인트들 간의 복잡한 유사성 관계를, 사람이 직관적으로 이해할 수 있는 2차원 지도에 시각화한다.
        - 이 지도를 통해 클러스터링 결과를 시각적으로 분석하고 평가할 수 있음.

In [ ]:
    # 5. t-SNE로 시각화를 위한 2차원 축소
    print("\n5. t-SNE를 사용하여 시각화를 위해 2차원으로 축소합니다...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    features_tsne = tsne.fit_transform(features_pca)
    print(f"t-SNE로 축소된 벡터의 형태: {features_tsne.shape}")

    # 6. 결과 시각화
    print("\n6. 결과를 시각화합니다...")
    visualize_clusters(features_tsne, true_labels, 't-SNE Visualization with True Labels')
    
    kmeans_labels_str = [f'Cluster {label}' for label in cluster_labels]
    visualize_clusters(features_tsne, kmeans_labels_str, 't-SNE Visualization with K-means Clusters')

 print("\n모든 작업이 완료되었습니다.")

# --- 3. 시각화 함수 ---
def visualize_clusters(tsne_results, labels, title):
    plt.figure(figsize=(10, 8))
    unique_labels = np.unique(labels)
    for label in unique_labels:
        indices = [i for i, l in enumerate(labels) if l == label]
        plt.scatter(tsne_results[indices, 0], tsne_results[indices, 1], label=label, alpha=0.7)
    plt.title(title)
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    plt.legend()
    plt.grid(True)
    plt.show()

### **분석 결과**

- **데이터 분포 확인 ('정답 지도'):**
    - 'Street'과 'Casual' 스타일은 데이터상에서 뚜렷한 군집을 형성하며, AI가 구분할 만한 고유한 시각적 특징을 가지고 있음을 확인.
    - 'Minimal' 스타일 또한 상대적으로 잘 뭉쳐있는 분포를 보였음.
    - 반면, 'Amekaji' 스타일은 다른 스타일들과 넓게 섞여 분포하여, 시각적으로 가장 특징이 모호한 스타일임을 파악함.

- **K-means 성능 평가 ('결과 지도'):**
    - K-means 알고리즘은 '정답 지도'의 데이터 분포를 매우 유사하게 재현해냈음.
    - 특히 'Street'과 'Casual', 'Minimal' 스타일이 뭉쳐있는 핵심 영역들을 성공적으로 각각 다른 클러스터로 분리해냈음.
    - 이는 우리 데이터가 정답 라벨 없이도 AI가 스스로 그룹을 나눌 수 있을 만큼 논리적인 시각적 패턴을 가지고 있음을 의미하는 매우 긍정적인 결과임.

### **결론 및 향후 프로젝트 기여 방안**
- 이번 K-means 클러스터링 분석은 최종 CNN 서비스 개발에 다음과 같이 기여할 수 있음.
    - **데이터 유효성 검증:** 우리가 사용할 이미지 데이터가 AI 모델 학습에 적합하며, 스타일 분류라는 목표를 달성할 수 있는 충분한 잠재력을 가지고 있음을 성공적으로 검증함.
    - **잠재적 난이도 예측:** 'Amekaji'처럼 다른 스타일과 시각적 특징이 많이 겹치는 데이터는 향후 CNN 모델이 분류하는 데 어려움을 겪을 수 있음을 미리 예측 가능함.
    - **모델 개발 방향 제시:** 향후 CNN 모델의 성능이 기대만큼 나오지 않거나 특정 스타일의 정확도가 유독 낮을 경우, 이번 분석 결과를 근거로 데이터의 근본적인 특징 때문일 수 있다는 빠른 진단이 가능. 이는 모델 구조 변경, 특정 스타일 데이터 추가 수집 등 구체적인 개선 전략을 세우는 데 중요한 기초 자료로 활용될 것임.
